In [1]:
# 长短期记忆网络
# - 忘记门: 将值朝0减少
# - 输入门: 决定不是忽略掉输入数据
# - 输出门: 决定是不是使用隐状态

# 门
# 输入是X_t, Hidden State H_t-1
# Input Gate: I_t = σ(X_t * W_xi + H_t-1 * W_hi + bi)
# Forgot Gate: F_t = σ(X_t * W_xf + H_t-1 * W_hf + bf)
# Output Gate: O_t = σ(X_t * W_xo + H_t-1 * W_ho + bo)

# 多了一个东西叫: 候选记忆单元
# c̅ = tanh(X_t * W_xc + H_t-1 * W_hc + bc)
# 唯一的区别是这里的单元没有用到任何gate

# 记忆单元
# 这里多了一个状态C, 也就是记忆单元Memory, 也随着时间变换
# C_t = F_t ⊙ C_t-1 + I_t ⊙ c̅_t
# F_t如果是0的话, 就说明要忘掉之前的一些记忆
# I_t如果是1的话, 就用这个候选记忆单元
# 区别: 这里是独立的, 就是又要前面的状态, 又要新的状态, 也可以都不要

# 隐状态
# H_t = O_t ⊙ tanh(C_t)
# 因为之前的C_t的两个门是独立的, 所以F_t ⊙ C_t-1和I_t ⊙ c̅_t两个[-1,1]之间合并就会变到[-2,2]
# (补充上面) 因为F_t和I_t是[0,1], 而C_t-1和c̅_t都是[-1, 1]
# 而tanh是把东西变到[-1,1]之间
# 我们用tanh把Memory C_t变到[-1, 1]之间
# O_t 是控制我们要不要输出. 如果是1就是要输出, 0就是当前和之前的Memory我都不需要

In [3]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)

In [10]:
# 初始化模型参数
def get_lstm_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    def four():
        return (normal(
            (num_inputs, num_hiddens)), normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    
    W_xi, W_hi, b_i = four()
    W_xf, W_hf, b_f = four()
    W_xo, W_ho, b_o = four()
    W_xc, W_hc, b_c = four()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

In [11]:
# 初始化函数
def init_lstm_state(batch_size, num_hiddens, device):
    # 这里H和C都需要初始化
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))

In [12]:
# 实际模型
def lstm(inputs, state, params):
    [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c, W_hq, b_q] = params
    (H, C) = state
    outputs = []
    
    for X in inputs:
        I = torch.sigmoid((X @ W_xi) + (H @ W_hi) + b_i)
        F = torch.sigmoid((X @ W_xf) + (H @ W_hf) + b_f)
        O = torch.sigmoid((X @ W_xo) + (H @ W_ho) + b_o)
        C_tilda = torch.tanh((X @ W_xc) + (H @ W_hc) + b_c)
        C = F * C + I * C_tilda
        H = O * torch.tanh(C)
        Y = (H @ W_hq) + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H, C)


In [ ]:
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_lstm_params,
                            init_lstm_state, lstm)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)

In [ ]:
# 简介实现
num_inputs = vocab_size
lstm_layer = nn.LSTM(num_inputs, num_hiddens)
model = d2l.RNNModel(lstm_layer, len(vocab))
model = model.to(device)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)